# horizon-decay: run on a real model

Does per-step error rise with depth (context degradation), or stay flat (pure compounding)?

**Before running:** add your API key in Colab's 🔑 Secrets panel (left sidebar) as `ANTHROPIC_API_KEY` or `OPENAI_API_KEY`, and turn on notebook access for it.

If your Anthropic key isn't scoped to a workspace, either create a key inside a workspace in the Anthropic Console, or also add a secret named `ANTHROPIC_WORKSPACE_ID` with your workspace ID.

In [ ]:
!git clone -q https://github.com/ev-prawda92/Horizon-Decay.git
%cd Horizon-Decay
!pip install -q -e ".[anthropic,openai]"

In [ ]:
import os
from google.colab import userdata

for k in ["ANTHROPIC_API_KEY", "ANTHROPIC_WORKSPACE_ID", "OPENAI_API_KEY"]:
    try:
        os.environ[k] = userdata.get(k)
    except Exception:
        pass

## Settings

`MODEL` is `anthropic:<model-id>` or `openai:<model-id>`; use any current model ID from the provider's docs. Start small (`TRIALS = 10`) to check cost and formatting, then scale to 30.

In [ ]:
MODEL = "anthropic:REPLACE_WITH_MODEL_ID"
LENGTHS = "8 16 32 64 128"
TRIALS = 10
CHUNK = 8

tag = MODEL.replace(':', '-').replace('/', '-')
OUT = f"results/{tag}"

## Sanity check: the pipeline on a simulated model

This costs nothing. The simulator has a known slope of 0.0006, so the `trace` slope CI below should contain it.

In [ ]:
!python -m horizon.run --model sim:0.005,0.0006 --lengths 16 64 --trials 30 --workers 1 --out /tmp/sim.jsonl
!python -m horizon.analyze /tmp/sim.jsonl --out /tmp/sim | head -12

## Run the real model

In [ ]:
!python -m horizon.run --model {MODEL} --lengths {LENGTHS} --trials {TRIALS} --chunk {CHUNK} --workers 4 --out {OUT}/trials.jsonl

## Analyze

In [ ]:
!python -m horizon.analyze {OUT}/trials.jsonl --out {OUT}

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{OUT}/*.png")):
    display(Image(p))

## Look at failures by hand

Always read some raw outputs. Format problems or a misread prompt can look like a real effect.

In [ ]:
import pandas as pd
from horizon.analyze import load
df = load(f"{OUT}/trials.jsonl")
bad = df[(df.condition == 'trace') & (~df.final_ok)].sort_values('n_steps')
print(len(bad), 'failed trace trials')
if len(bad):
    r = bad.iloc[0]
    print('first error at step', r.first_error, 'of', r.n_steps)
    print(r.raw[:1500])

## Save results

Download `results/` and commit it to the repo, or push from here with a GitHub token.

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')